# PUBG API Data Extraction & Structuring
This notebook tests the extraction of PUBG match data and structures it into flattened DataFrames suitable for a data pipeline.

In [ ]:
import requests
import pandas as pd
from config import api_key
from IPython.display import display

In [ ]:
def get_pubg_samples(api_key, shard='steam'):
    """Fetch a list of sample matches from the PUBG API."""
    url = f"https://api.pubg.com/shards/{shard}/samples"
    headers = {
        "Authorization": f"Bearer {api_key}", 
        "Accept": "application/vnd.api+json"
    }
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None

def get_match_details(api_key, match_id, shard='steam'):
    """Fetch detailed information for a specific match."""
    url = f"https://api.pubg.com/shards/{shard}/matches/{match_id}"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Accept": "application/vnd.api+json"
    }
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None

In [ ]:
# 1. Fetch Sample Matches
samples_data = get_pubg_samples(api_key)
if samples_data:
    matches_list = samples_data['data']['relationships']['matches']['data']
    print(f"Found {len(matches_list)} sample matches.")
    first_match_id = matches_list[0]['id']
    print(f"Selected first match ID: {first_match_id}")

In [ ]:
# 2. Fetch Match Details
match_json = get_match_details(api_key, first_match_id)
if match_json:
    print(f"Successfully retrieved details for match {first_match_id}")

In [ ]:
def parse_match_data(match_json):
    """
    Parses a PUBG match JSON response into separate flattened DataFrames.
    Returns a dictionary of DataFrames: match, participants, rosters, assets.
    """
    # 1. Extract Core Match Data
    match_data = match_json['data']
    df_match = pd.json_normalize(match_data)
    # Flatten attributes and remove prefixes
    df_match.columns = [col.replace('attributes.', '').replace('relationships.', '') for col in df_match.columns]
    
    # 2. Extract Included Objects (Participants, Rosters, Assets)
    included = match_json.get('included', [])
    
    participants = [item for item in included if item['type'] == 'participant']
    rosters = [item for item in included if item['type'] == 'roster']
    assets = [item for item in included if item['type'] == 'asset']
    
    # Flatten Participants
    df_participants = pd.json_normalize(participants)
    if not df_participants.empty:
        df_participants.columns = [col.replace('attributes.', '').replace('stats.', '').replace('relationships.', '') for col in df_participants.columns]
        # Add match_id for reference
        df_participants['match_id'] = match_data['id']
    
    # Flatten Rosters
    df_rosters = pd.json_normalize(rosters)
    if not df_rosters.empty:
        df_rosters.columns = [col.replace('attributes.', '').replace('relationships.', '') for col in df_rosters.columns]
        df_rosters['match_id'] = match_data['id']
    
    # Flatten Assets
    df_assets = pd.json_normalize(assets)
    if not df_assets.empty:
        df_assets.columns = [col.replace('attributes.', '').replace('relationships.', '') for col in df_assets.columns]
        df_assets['match_id'] = match_data['id']
        
    return {
        'match': df_match,
        'participants': df_participants,
        'rosters': df_rosters,
        'assets': df_assets
    }

dfs = parse_match_data(match_json)

In [ ]:
print("--- Match Overview ---")
display(dfs['match'].head())

print("\n--- Participants (Player Stats) ---")
display(dfs['participants'].head())

print("\n--- Rosters (Teams) ---")
display(dfs['rosters'].head())

print("\n--- Assets (Telemetry) ---")
display(dfs['assets'].head())